In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ml-ware-26-sherlock-files/sample_submission.csv
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train_labels.json
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/test/video_5704.mp4
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/test/video_5624.mp4
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/test/video_5719.mp4
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/test/video_5856.mp4
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/test/video_5854.mp4
/kaggle/input/competitions/ml-ware-26-sherlock-files/Evid

In [2]:
import os

# Get the contents of the current working directory
directory_contents = os.listdir('/kaggle/input/datasets/agasthidoshi/features/kaggle/working/features')

print("Contents of the current directory:")
for item in directory_contents:
    print(item)

# To list the contents of a specific directory, provide the path as an argument
# Example: os.listdir('/path/to/your/directory')

Contents of the current directory:
video_4322.npy
video_4829.npy
video_1312.npy
video_569.npy
video_3742.npy
video_4015.npy
video_2537.npy
video_2774.npy
video_1159.npy
video_5529.npy
video_4060.npy
video_4554.npy
video_1844.npy
video_5481.npy
video_2749.npy
video_4546.npy
video_2278.npy
video_2442.npy
video_2997.npy
video_1050.npy
video_935.npy
video_2722.npy
video_383.npy
video_3949.npy
video_4559.npy
video_1062.npy
video_3169.npy
video_3589.npy
video_3392.npy
video_5363.npy
video_856.npy
video_2811.npy
video_1472.npy
video_1149.npy
video_4419.npy
video_498.npy
video_1265.npy
video_3721.npy
video_2638.npy
video_5453.npy
video_3502.npy
video_3119.npy
video_5055.npy
video_3197.npy
video_5221.npy
video_5236.npy
video_2323.npy
video_3200.npy
video_664.npy
video_754.npy
video_5156.npy
video_3916.npy
video_700.npy
video_930.npy
video_2464.npy
video_1356.npy
video_3517.npy
video_2418.npy
video_2159.npy
video_2744.npy
video_4991.npy
video_4182.npy
video_2795.npy
video_4023.npy
video_5368.npy

In [3]:
import os
print(len(os.listdir("/kaggle/input/datasets/agasthidoshi/features/kaggle/working/features")))

5611


In [4]:
import os, json, csv, time, warnings
import numpy as np
import cv2
from scipy.stats import kendalltau
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
warnings.filterwarnings("ignore")

In [5]:
try:
    import torch
    import torch.nn as nn
    import torchvision.models as models
    import torchvision.transforms as transforms
    TORCH_AVAILABLE = True
    print("✓ PyTorch available — using ResNet18 deep features")
except ImportError:
    TORCH_AVAILABLE = False
    print("✗ PyTorch not found — using HOG+flow classical features")

✓ PyTorch available — using ResNet18 deep features


In [6]:
CFG = dict(
    train_dir   = "/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train",
    test_dir    = "/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/test",
    label_file  = "/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train_labels.json",
    output_csv  = "/kaggle/working/submission.csv",

    # Feature extraction
    img_size_deep    = 112,   # ResNet18 input (we use 112 for speed)
    img_size_shallow = 64,    # HOG input
    hog_cell         = 8,     # HOG cell size (pixels)
    feat_dim_cap     = 128,   # Keep first D dims of frame feat for pair feature
    flow_bins        = 8,     # Angular histogram bins for flow

    # Training
    pairs_per_video  = 60,   # Pairwise samples per training video
    max_train_videos = None,  # None = use all
    rf_trees         = 30,
    rf_max_depth     = 8,
    rf_jobs          = -1,

    # Inference
    max_pairs_test   = 1500,  # Cap pairs for large videos at test time
    smooth_iters     = 3,     # Smoothness refinement passes

    # Validation
    val_videos       = 200,   # Last N training videos held out for τ estimate
)

In [7]:
if TORCH_AVAILABLE:
    class DeepFrameEncoder:
        """Extract 512-d features from frames using pretrained ResNet18."""
        def __init__(self, device=None):
            if device is None:
                self.device = "cuda" if torch.cuda.is_available() else "cpu"
            else:
                self.device = device
            model = models.resnet18(pretrained=True)
            self.model = nn.Sequential(*list(model.children())[:-1])
            self.model.eval().to(self.device)
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((CFG["img_size_deep"], CFG["img_size_deep"])),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485,0.456,0.406],
                                     std=[0.229,0.224,0.225]),
            ])
            print(f"  ResNet18 encoder on {self.device}")

        @torch.no_grad()
        def encode_batch(self, frames_bgr, batch_size=64):
            """Encode a list of BGR frames -> numpy (N, 512)."""
            tensors = [self.transform(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
                       for f in frames_bgr]
            feats = []
            for i in range(0, len(tensors), batch_size):
                batch = torch.stack(tensors[i:i+batch_size]).to(self.device)
                out = self.model(batch).squeeze(-1).squeeze(-1)
                feats.append(out.cpu().numpy())
            return np.vstack(feats)
else:
    class DeepFrameEncoder:
        def __init__(self, *a, **kw): raise RuntimeError("PyTorch not available")

In [8]:
def extract_hog_features(frames_bgr):
    """
    Extract classical physics-aware per-frame features.
    Returns (N, D) array.

    Features per frame:
      • HOG 64×64 with 8-px cells  → 324-d  (edge/shape structure)
      • HSV colour histogram 16+8+8 → 32-d   (appearance)
      • Laplacian variance          →  1-d   (sharpness/blur)
      Total                        → 357-d
    """
    W = H = CFG["img_size_shallow"]
    hog_desc = cv2.HOGDescriptor(
        (W, H),           # win_size
        (2*CFG["hog_cell"], 2*CFG["hog_cell"]),  # block_size
        (CFG["hog_cell"],   CFG["hog_cell"]),     # block_stride
        (CFG["hog_cell"],   CFG["hog_cell"]),     # cell_size
        9                                          # nbins
    )

    all_feats = []
    for frame in frames_bgr:
        small = cv2.resize(frame, (W, H))
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        hog_f = hog_desc.compute(small).flatten()

        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        h_hist = cv2.calcHist([hsv],[0],None,[16],[0,180]).flatten()
        s_hist = cv2.calcHist([hsv],[1],None,[8], [0,256]).flatten()
        v_hist = cv2.calcHist([hsv],[2],None,[8], [0,256]).flatten()
        for h in (h_hist, s_hist, v_hist):
            h /= (h.sum() + 1e-7)

        lap_var = np.float32(cv2.Laplacian(gray, cv2.CV_32F).var())

        feat = np.concatenate([hog_f, h_hist, s_hist, v_hist, [lap_var]])
        all_feats.append(feat.astype(np.float32))

    return np.stack(all_feats)


In [9]:
def extract_flow_features(frames_bgr, size=64):
    """
    Compute per-adjacent-pair motion feature vectors (N-1, 15).

    Features:
      mean / std / max flow magnitude           — motion intensity
      divergence proxy (∂vx/∂x + ∂vy/∂y)      — expanding/contracting
      frame-diff mean & std                     — pixel-level change
      angular histogram (8 bins)                — dominant motion direction

    These encode physics: gravity pulls down, projectiles arc, etc.
    """
    flow_feats = []
    prev_gray = None
    for frame in frames_bgr:
        gray = cv2.cvtColor(cv2.resize(frame, (size, size)),
                            cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=2, winsize=13,
                iterations=2, poly_n=5, poly_sigma=1.1, flags=0)
            mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])

            div_x = np.gradient(flow[...,0], axis=1).mean()
            div_y = np.gradient(flow[...,1], axis=0).mean()

            diff = cv2.absdiff(prev_gray, gray).astype(np.float32)

            ang_hist, _ = np.histogram(ang.flatten(), bins=CFG["flow_bins"],
                                       range=(0, 2*np.pi))
            ang_hist = ang_hist.astype(np.float32) / (ang_hist.sum() + 1e-7)

            fv = np.array([
                mag.mean(), mag.std(), mag.max(),
                div_x, div_y,
                diff.mean(), diff.std()
            ], dtype=np.float32)
            flow_feats.append(np.concatenate([fv, ang_hist]))
        prev_gray = gray

    if not flow_feats:
        return np.zeros((0, 7 + CFG["flow_bins"]), dtype=np.float32)
    return np.stack(flow_feats)   # (N-1, 15)

In [10]:
_FLOW_DIM = 7 + CFG["flow_bins"]   # 15

def build_pair_feature(feat_i, feat_j, flow_i=None):
    """
    Build fixed-dim pairwise feature for (i, j).
    The feature is ASYMMETRIC so the model can learn temporal direction.

      [feat_i[:D] | feat_j[:D] | (feat_i−feat_j)[:D] | |feat_i−feat_j|[:D] | flow_i]

    D = CFG["feat_dim_cap"] (128 by default)
    Total dim = 4*D + 15 = 527
    """
    D = CFG["feat_dim_cap"]
    d = feat_i - feat_j
    pair = np.concatenate([feat_i[:D], feat_j[:D], d[:D], np.abs(d)[:D]])
    flow_vec = flow_i if flow_i is not None else np.zeros(_FLOW_DIM, dtype=np.float32)
    return np.concatenate([pair, flow_vec]).astype(np.float32)

In [11]:
def load_frames(path, max_frames=80):
    cap = cv2.VideoCapture(path)
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


def get_all_features(frames, encoder=None):
    """
    Get per-frame features + flow features for a list of BGR frames.
    Uses deep encoder if provided, else HOG.
    """
    if encoder is not None and TORCH_AVAILABLE:
        frame_feats = encoder.encode_batch(frames)
    else:
        frame_feats = extract_hog_features(frames)
    flow_feats = extract_flow_features(frames)
    return frame_feats, flow_feats

In [12]:
def _build_rank_array(label_list, N):
    """
    label_list[r] = corrupted_frame_index  →  that frame is at temporal rank r.
    Returns rank array: rank[corrupted_idx] = temporal_rank.
    """
    rank = np.arange(N, dtype=np.int32)   # default: identity
    for r, ci in enumerate(label_list):
        if ci < N:
            rank[ci] = r
    return rank


def generate_pairs(video_path, label, n_pairs=400, seed=42, encoder=None):
    """
    Generate n_pairs balanced (positive, negative) training examples.
    Returns X (list of feat vectors), y (list of 0/1).
    """
    rng = np.random.RandomState(seed)
    frames = load_frames(video_path)
    if len(frames) < 3:
        return [], []

    N = len(frames)
    frame_feats, flow_feats = get_all_features(frames, encoder)

    rank = _build_rank_array(label, N)
    valid_len = min(len(label), N)
    valid_idx = list(range(valid_len))

    half = n_pairs // 2
    X, y = [], []
    pos_n = neg_n = 0

    attempts = 0
    while (pos_n < half or neg_n < half) and attempts < n_pairs * 20:
        attempts += 1
        r, s = rng.choice(valid_idx, 2, replace=False)
        ci = label[r] if r < len(label) else r
        cj = label[s] if s < len(label) else s
        if ci >= N or cj >= N:
            continue

        ri, rj = rank[ci], rank[cj]
        if ri == rj:
            continue

        flow_i = flow_feats[ci - 1] if (ci > 0 and ci - 1 < len(flow_feats)) else None
        feat = build_pair_feature(frame_feats[ci], frame_feats[cj], flow_i)
        lbl  = 1 if ri < rj else 0

        if lbl == 1 and pos_n < half:
            X.append(feat); y.append(1); pos_n += 1
        elif lbl == 0 and neg_n < half:
            X.append(feat); y.append(0); neg_n += 1

    return X, y

In [13]:
def train(encoder=None):

    print("\n" + "="*60)
    print("PHASE 1 — Loading labels")
    print("="*60)

    with open(CFG["label_file"]) as f:
        labels = json.load(f)

    video_ids = sorted(labels.keys())

    # limit to 3000 videos
    video_ids = video_ids[:800]

    if CFG["max_train_videos"]:
        video_ids = video_ids[:CFG["max_train_videos"]]

    val_ids = set(video_ids[-CFG["val_videos"]:])
    train_ids = [v for v in video_ids if v not in val_ids]

    print(f"  Train videos: {len(train_ids)}   Val videos: {len(val_ids)}")

    print("\n" + "="*60)
    print("PHASE 2 — Generating pairwise training data")
    print("="*60)

    all_X, all_y = [], []
    t0 = time.time()

    for k, vid in enumerate(train_ids):

        path = os.path.join(CFG["train_dir"], f"{vid}.mp4")

        if not os.path.exists(path):
            continue

        try:
            X, y = generate_pairs(
                path,
                labels[vid],
                n_pairs=CFG["pairs_per_video"],
                seed=k,
                encoder=encoder
            )

            all_X.extend(X)
            all_y.extend(y)

        except Exception:
            continue

        if (k + 1) % 200 == 0:

            elapsed = time.time() - t0
            eta = elapsed / (k + 1) * (len(train_ids) - k - 1)

            print(
                f"  [{k+1:>5}/{len(train_ids)}]  pairs={len(all_X):>8}"
                f"  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m"
            )

    print(f"\n  Total pairs  : {len(all_X)}")
    print(f"  Class balance: {sum(all_y)}/{len(all_y)} positive")

    print("\n" + "="*60)
    print("PHASE 3 — Training pairwise RandomForest")
    print("="*60)

    X_arr = np.array(all_X, dtype=np.float32)
    y_arr = np.array(all_y, dtype=np.int32)

    model = Pipeline([
        ("sc", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=CFG["rf_trees"],
            max_depth=CFG["rf_max_depth"],
            min_samples_leaf=5,
            n_jobs=CFG["rf_jobs"],
            random_state=42
        ))
    ])

    t_train = time.time()
    model.fit(X_arr, y_arr)

    print(f"  Done in {time.time()-t_train:.1f}s")

    acc = (model.predict(X_arr[:2000]) == y_arr[:2000]).mean()
    print(f"  Train accuracy (2k sample): {acc:.4f}")

    return model, val_ids, labels

In [14]:
def reconstruct_order(frame_feats, flow_feats, model, max_pairs=None):

    N = len(frame_feats)

    if N == 1:
        return [0]

    if N == 2:
        feat = build_pair_feature(
            frame_feats[0],
            frame_feats[1],
            flow_feats[0] if len(flow_feats) > 0 else None
        )

        p = model.predict_proba([feat])[0][1]

        return [0, 1] if p >= 0.5 else [1, 0]

    all_pairs = [(i, j) for i in range(N) for j in range(i+1, N)]

    cap = max_pairs or CFG["max_pairs_test"]

    if len(all_pairs) > cap:

        rng = np.random.RandomState(0)
        idx = rng.choice(len(all_pairs), cap, replace=False)
        all_pairs = [all_pairs[k] for k in idx]

    feats = []

    for i, j in all_pairs:

        fi = flow_feats[i-1] if i > 0 and i-1 < len(flow_feats) else None

        feats.append(
            build_pair_feature(
                frame_feats[i],
                frame_feats[j],
                fi
            )
        )

    feats = np.array(feats, dtype=np.float32)

    probs = model.predict_proba(feats)[:, 1]

    scores = np.zeros(N)

    for k, (i, j) in enumerate(all_pairs):

        margin = probs[k] - 0.5
        scores[i] += margin
        scores[j] -= margin

    return np.argsort(-scores).tolist()

In [15]:
def smooth_order(order, frame_feats, n_iters=3):

    order = list(order)
    N = len(order)

    if N < 4:
        return order

    def neighbor_sim(order, pos):

        i = order[pos]
        s = 0

        if pos > 0:
            j = order[pos-1]
            fi, fj = frame_feats[i], frame_feats[j]
            s += fi.dot(fj) / (np.linalg.norm(fi)*np.linalg.norm(fj) + 1e-8)

        if pos < N-1:
            j = order[pos+1]
            fi, fj = frame_feats[i], frame_feats[j]
            s += fi.dot(fj) / (np.linalg.norm(fi)*np.linalg.norm(fj) + 1e-8)

        return s

    for _ in range(n_iters):

        improved = False

        for pos in range(N-1):

            before = neighbor_sim(order, pos) + neighbor_sim(order, pos+1)

            order[pos], order[pos+1] = order[pos+1], order[pos]

            after = neighbor_sim(order, pos) + neighbor_sim(order, pos+1)

            if after > before:
                improved = True
            else:
                order[pos], order[pos+1] = order[pos+1], order[pos]

        if not improved:
            break

    return order

In [16]:
def validate(model, val_ids, labels, encoder=None):

    print("\n" + "="*60)
    print("PHASE 4 — Validation (Kendall Tau)")
    print("="*60)

    taus = []

    for vid in sorted(val_ids):

        path = os.path.join(CFG["train_dir"], f"{vid}.mp4")

        if not os.path.exists(path):
            continue

        frames = load_frames(path)

        if len(frames) < 3:
            continue

        try:

            N = len(frames)

            frame_feats, flow_feats = get_all_features(frames, encoder)

            raw_order = reconstruct_order(frame_feats, flow_feats, model)

            pred_order = smooth_order(raw_order, frame_feats, CFG["smooth_iters"])

            pred_rank = np.zeros(N)

            for r, ci in enumerate(pred_order):
                pred_rank[ci] = r

            true_rank = np.full(N, -1)

            label_arr = labels[vid]

            for r, ci in enumerate(label_arr[:N]):

                if ci < N:
                    true_rank[ci] = r

            mask = true_rank >= 0

            if mask.sum() < 2:
                continue

            tau, _ = kendalltau(pred_rank[mask], true_rank[mask])

            if np.isnan(tau):
                continue

            print(f"{vid} -> tau = {tau:.3f}")

            taus.append(tau)

        except Exception:

            print(f"{vid} -> ERROR")

    if len(taus) > 0:

        taus = np.array(taus)

        print("\nValidation Summary")
        print("-"*40)

        print(f"Videos evaluated : {len(taus)}")
        print(f"Mean Kendall τ   : {np.mean(taus):.4f}")
        print(f"Std              : {np.std(taus):.4f}")
        print(f"Min / Max        : {np.min(taus):.4f} / {np.max(taus):.4f}")
        print(f"τ > 0.5 fraction : {(taus > 0.5).mean():.3f}")

        return float(np.mean(taus))

    else:

        print("No validation results.")

        return 0.0

In [17]:
def generate_submission(model, encoder=None):
    print("\n" + "="*60)
    print("PHASE 5 — Test Inference")
    print("="*60)
    test_files = sorted(f for f in os.listdir(CFG["test_dir"])
                        if f.endswith(".mp4"))
    print(f"  Test videos: {len(test_files)}")

    rows = []
    t0 = time.time()
    for k, fname in enumerate(test_files):
        vid_id = fname.replace(".mp4", "")
        path   = os.path.join(CFG["test_dir"], fname)
        try:
            frames = load_frames(path)
            N = len(frames)
            if N == 0:
                rows.append((vid_id, "0")); continue

            frame_feats, flow_feats = get_all_features(frames, encoder)
            raw   = reconstruct_order(frame_feats, flow_feats, model)
            order = smooth_order(raw, frame_feats, CFG["smooth_iters"])
            rows.append((vid_id, " ".join(map(str, order))))
        except Exception as e:
            frames_n = len(frames) if "frames" in dir() else 10
            rows.append((vid_id, " ".join(map(str, range(frames_n)))))

        if (k + 1) % 20 == 0:
            print(f"  [{k+1}/{len(test_files)}]  {time.time()-t0:.1f}s elapsed")

    with open(CFG["output_csv"], "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["video_id", "order"])
        w.writerows(rows)

    print(f"\n  Submission saved → {CFG['output_csv']}")
    print(f"  Total time       : {(time.time()-t0)/60:.1f} min")
    return rows

In [18]:
def main():
    print("=" * 60)
    print("MLWare '26 Sherlock Files — OPN Pairwise Ranking Pipeline")
    print("=" * 60)
    t_total = time.time()

    # Initialise deep encoder if available
    encoder = DeepFrameEncoder() if TORCH_AVAILABLE else None

    # Train
    model, val_ids, labels = train(encoder=encoder)

    # Validate
    tau = validate(model, val_ids, labels, encoder=encoder)

    # Submit
    generate_submission(model, encoder=encoder)

    print("\n" + "=" * 60)
    print(f"ALL DONE  |  Kendall τ ≈ {tau:.4f}  |  "
          f"Total time: {(time.time()-t_total)/60:.1f} min")
    print("=" * 60)


if __name__ == "__main__":
    main()

MLWare '26 Sherlock Files — OPN Pairwise Ranking Pipeline
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 206MB/s]


  ResNet18 encoder on cuda

PHASE 1 — Loading labels
  Train videos: 600   Val videos: 200

PHASE 2 — Generating pairwise training data
  [  200/600]  pairs=   11958  elapsed=0.8m  ETA=1.7m
  [  400/600]  pairs=   23734  elapsed=1.7m  ETA=0.9m
  [  600/600]  pairs=   35689  elapsed=2.6m  ETA=0.0m

  Total pairs  : 35689
  Class balance: 17841/35689 positive

PHASE 3 — Training pairwise RandomForest
  Done in 7.3s
  Train accuracy (2k sample): 0.7900

PHASE 4 — Validation (Kendall Tau)
video_1538 -> tau = -0.240
video_1539 -> tau = 0.214
video_154 -> tau = 0.262
video_1540 -> tau = 0.222
video_1541 -> tau = -0.013
video_1542 -> tau = -0.046
video_1543 -> tau = -0.249
video_1544 -> tau = -0.014
video_1545 -> tau = -0.140
video_1546 -> tau = -0.101
video_1547 -> tau = -0.287
video_1548 -> tau = -0.160
video_1549 -> tau = 0.216
video_155 -> tau = 0.417
video_1550 -> tau = 0.244
video_1551 -> tau = 0.300
video_1552 -> tau = -0.027
video_1553 -> tau = -0.191
video_1554 -> tau = 0.267
video_1